# 项目 2：经典造父变星 RT Aur — 周期折叠与距离交叉验证

流程：TESS 光变（myastro.io，坐标随 meta 返回）→ LS 粗搜 + **PDM 精化** → 折叠 →
P-L 关系估距离 → Gaia DR3 视差独立验证。

> 数据源备注：ASAS-SN v2 后端当前失联（DNS NXDOMAIN，见 `io/sources/asassn.py` 头注），
> 光变改用 TESS。周期分析方法论的坑：造父是**非正弦**锯齿光变，
> LS（单正弦拟合）会系统性偏峰，须用 PDM（相位离散度极小化）精化。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Arial Unicode MS", "Heiti TC"]

from myastro import lc
from myastro.io.sources.tess import tess_lightcurve
from myastro.io.sources.gaia import gaia_cone

P_LIT = 3.728354        # 文献周期 (d)，V 波段 4.55–5.66

result = tess_lightcurve("RT Aur", sector=20, cadence="2m")
RA, DEC = result.meta["ra"], result.meta["dec"]     # 从 FITS 头拿精确坐标
print(f"cache_hit: {result.cache_hit} | 目标坐标 ({RA:.5f}, {DEC:+.5f})")

df = pd.read_parquet(result.paths[0])
df = lc.quality_cut(df)
t = df.time.values
f = lc.normalize(df.flux.values)
print(f"{len(t)} cadences, {t.max()-t.min():.1f} 天基线")


In [ ]:
# Gaia：锥心最近的源 = RT Aur 本体（用 TESS 精确坐标，不再手抄）
gres = gaia_cone(RA, DEC, radius_deg=0.02)
gdf = pd.read_parquet(gres.paths[0])
d2 = (gdf.ra - RA)**2 + np.cos(np.deg2rad(DEC))**2 * (gdf.dec - DEC)**2
star = gdf.iloc[int(d2.values.argmin())]
assert star.phot_g_mean_mag < 10, "交叉匹配到暗星，坐标或锥半径有问题"
plx, e_plx, G = star.parallax, star.parallax_error, star.phot_g_mean_mag
print(f"Gaia: parallax = {plx:.3f} ± {e_plx:.3f} mas, G = {G:.2f}")
d_gaia_pc = 1000 / plx
print(f"视差距离 ≈ {d_gaia_pc:.0f} pc")

## 1. 周期：LS 粗搜 + PDM 精化

In [ ]:
# 周期搜索：LS 粗搜 + PDM 精化（库内一个函数完成两步）
res = lc.period_search(t, f, min_period=1.0, max_period=30.0)
P_ls, P_pdm = res.coarse_period, res.best_period
print(f"LS 粗搜  P = {P_ls:.5f} d（vs 文献 {P_LIT}，偏差 "
      f"{abs(P_ls-P_LIT)/P_LIT*100:.2f}%——非正弦偏峰）")
print(f"PDM 精化 P = {P_pdm:.6f} d（偏差 {abs(P_pdm-P_LIT)/P_LIT*100:.3f}%）")
P = P_pdm


In [ ]:
# 为什么两步：LS 用单正弦拟合，对造父这类非正弦光变系统性偏峰。
print(f"PDM 相对 LS 修正 {res.correction_pct:.2f}%（方法={res.method}）")
assert abs(P_pdm - P_LIT) <= abs(P_ls - P_LIT), "精化未改善，检查数据"


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

# 左：LS 周期图（库返回的 periodogram）
axes[0].plot(res.periods, res.power, lw=0.6)
axes[0].axvline(res.coarse_period, color="r", ls="--",
                label=f"LS {res.coarse_period:.4f}")
axes[0].axvline(P_LIT, color="g", ls=":", label=f"文献 {P_LIT}")
axes[0].set(xlabel="周期 (d)", ylabel="LS power", title="LS 周期图")
axes[0].legend()

# 右：PDM θ 曲线（库内精化时一并算出，直接画，无需重算）
axes[1].plot(res.refine_periods, res.refine_theta, lw=0.8)
axes[1].axvline(res.best_period, color="r", ls="--",
                label=f"PDM {res.best_period:.5f}")
axes[1].set(xlabel="周期 (d)", ylabel="PDM θ", title="PDM 精化")
axes[1].legend()
fig.tight_layout(); fig.savefig("out/02-period.png", dpi=150); fig


## 2. 相位折叠

造父 = 径向脉动：上升支陡、下降支缓的不对称锯齿（对比凌星的盒状凹陷）。

In [ ]:
# 折叠：[0,1) 约定（周期变星相位从极小时刻起算）
phase, f_sorted = lc.fold(t, f, P, centered=False)
bc, bm = lc.bin_phase(phase, f_sorted, nbins=30)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(phase, f_sorted, ".", ms=1, alpha=0.15)
ax.plot(bc, bm, "r-", lw=2, label="bin 中位数")
ax.set(xlabel="相位", ylabel="归一化通量", title=f"RT Aur 折叠 (P = {P:.5f} d)")
ax.legend(); fig.tight_layout(); fig.savefig("out/02-fold.png", dpi=150); fig

amp_mag = lc.mag_amplitude(f)
print(f"峰谷幅度 ≈ {amp_mag:.2f} mag（文献 ~1.1 mag）")


## 3. P-L 关系估距离 vs Gaia 视差

V 波段周光关系（银河系经典造父，Ripepi+2019 简化）：
$M_V = -2.76\,\log_{10} P - 1.40$

G→V 色差为粗近似（教学级，误差 ~0.3 mag → 距离 ~15%）。

In [ ]:
M_V = -2.76*np.log10(P) - 1.40
m_V = G + 0.35
mu = m_V - M_V
d_pl_pc = 10**((mu+5)/5)
print(f"P-L: M_V = {M_V:.2f}, m_V ≈ {m_V:.2f}, μ = {mu:.2f}")
print(f"P-L 距离     ≈ {d_pl_pc:.0f} pc")
print(f"Gaia 视差距离 ≈ {d_gaia_pc:.0f} pc")
print(f"比值 = {d_pl_pc/d_gaia_pc:.2f}（色差近似误差内即吻合）")

## 4. 结论

- LS 偏峰 +1.17%（非正弦系统性），PDM 精化到 0.59%——方法论教训：**周期工具要匹配光变形状**；两步流程已固化为 `myastro.lc.period_search`
- 折叠轮廓呈典型造父不对称锯齿，TESS 波段幅度 ~0.51 mag（TESS 比 V 红，幅度天然压缩；文献 V 波段 ~1.1 mag）
- P-L 距离与 Gaia 视差同量级吻合（教学精度）：542 pc vs 551 pc
- 全程 myastro.io：TESS 光变、Gaia 星表均入库，二次运行零网络（`cache_hit: True` 且 meta 坐标随缓存返回）

### 技术要点

- 相位约定：`lc.fold(..., centered=False)` → [0,1) 给周期变星；凌星用默认 `centered=True` + 目录 T0
- `lc.bin_phase` 自动识别相位区间，与 fold 的两种约定配套
- PDM 精化网格：`res.refine_periods/refine_theta` 随结果返回，可直接画 θ 曲线（早期版本 notebook 自己重算）
